In [2]:
# %%
import json
import joblib
from pathlib import Path

MODEL_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_models")

model_path = MODEL_DIR / "best_binary_segA_vs_segBC.joblib"
metadata_path = MODEL_DIR / "model_metadata.json"

best_model = joblib.load(model_path)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

metadata

{'best_model_name': 'HistGradientBoosting',
 'best_threshold': 0.45,
 'task': 'SEG_A vs SEG_BC',
 'label_mapping': {'0': 'SEG_A', '1': 'SEG_BC'},
 'test_metrics': {'model': 'HistGradientBoosting',
  'threshold': 0.45,
  'accuracy': 0.7722689075630252,
  'macro_f1': 0.7703775750550399,
  'weighted_f1': 0.7719711978163979,
  'roc_auc': 0.8518300292864351,
  'SEG_A_precision': 0.7809885931558935,
  'SEG_A_recall': 0.8017174082747853,
  'SEG_A_f1': 0.7912172573189522,
  'SEG_BC_precision': 0.7615023474178404,
  'SEG_BC_recall': 0.737943585077343,
  'SEG_BC_f1': 0.7495378927911276}}

In [3]:
# %%
best_threshold = metadata["best_threshold"]
best_model_name = metadata["best_model_name"]

print("Loaded model:", best_model_name)
print("Decision threshold:", best_threshold)

Loaded model: HistGradientBoosting
Decision threshold: 0.45


In [4]:
# %%
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
from pathlib import Path

TENSOR_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_tensors")

X_torch = torch.load(TENSOR_DIR / "X_features.pt")
y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
fold_torch = torch.load(TENSOR_DIR / "folds.pt")

X_tf = tf.convert_to_tensor(X_torch.cpu().numpy(), dtype=tf.float32)
y_tf = tf.convert_to_tensor(y_torch.cpu().numpy(), dtype=tf.int64)
fold_tf = tf.convert_to_tensor(fold_torch.cpu().numpy(), dtype=tf.int64)

print("X:", X_tf.shape)
print("y:", y_tf.shape)
print("folds:", fold_tf.shape)

C:\Users\omarl\AppData\Local\Temp\ipykernel_68556\840025626.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_torch = torch.load(TENSOR_DIR / "X_features.pt")


X: (20931, 86, 65)
y: (20931,)
folds: (20931,)


C:\Users\omarl\AppData\Local\Temp\ipykernel_68556\840025626.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
C:\Users\om

In [5]:
# %%
labeled_mask = y_tf != -1

X_labeled = tf.boolean_mask(X_tf, labeled_mask)
y_labeled = tf.boolean_mask(y_tf, labeled_mask)
fold_labeled = tf.boolean_mask(fold_tf, labeled_mask)

print("X labeled:", X_labeled.shape)
print("y labeled:", y_labeled.shape)

X labeled: (11899, 86, 65)
y labeled: (11899,)


In [6]:
# %%
test_fold = 3

test_mask = fold_labeled == test_fold

X_test = tf.boolean_mask(X_labeled, test_mask)
y_test = tf.boolean_mask(y_labeled, test_mask)
fold_test = tf.boolean_mask(fold_labeled, test_mask)

print("Test:", X_test.shape, y_test.shape)

Test: (2380, 86, 65) (2380,)


In [22]:
# %%
unlabeled_mask = y_tf == -1

X_unlabeled = tf.boolean_mask(X_tf, unlabeled_mask)
y_unlabeled = tf.boolean_mask(y_tf, unlabeled_mask)
fold_unlabeled = tf.boolean_mask(fold_tf, unlabeled_mask)

print("Unlabeled X:", X_unlabeled.shape)
print("Unlabeled y:", y_unlabeled.shape)
print("Unlabeled folds:", fold_unlabeled.shape)

Unlabeled X: (9032, 86, 65)
Unlabeled y: (9032,)
Unlabeled folds: (9032,)


In [23]:
# %%
X_unlabeled_flat = X_unlabeled.numpy().reshape(X_unlabeled.shape[0], -1)

print("X_unlabeled_flat:", X_unlabeled_flat.shape)

X_unlabeled_flat: (9032, 5590)


In [24]:
# %%
unlabeled_proba_bc = best_model.predict_proba(X_unlabeled_flat)[:, 1]

unlabeled_pred_binary = (unlabeled_proba_bc >= best_threshold).astype(int)

unlabeled_pred_label = np.where(
    unlabeled_pred_binary == 0,
    "SEG_A",
    "SEG_BC"
)

unlabeled_predictions_df = pd.DataFrame({
    "true_label_encoded": y_unlabeled.numpy(),   # should be -1
    "prob_SEG_A": 1 - unlabeled_proba_bc,
    "prob_SEG_BC": unlabeled_proba_bc,
    "pred_binary_encoded": unlabeled_pred_binary,
    "pred_binary_label": unlabeled_pred_label,
    "decision_threshold": best_threshold,
    "hcp_fold": fold_unlabeled.numpy(),
    "model_name": best_model_name
})

unlabeled_predictions_df.head()

,true_label_encoded,prob_SEG_A,prob_SEG_BC,pred_binary_encoded,pred_binary_label,decision_threshold,hcp_fold,model_name
0,-1,0.889189,0.110811,0,SEG_A,0.45,-1,HistGradientBoosting
1,-1,0.909169,0.090831,0,SEG_A,0.45,-1,HistGradientBoosting
2,-1,0.910450,0.089550,0,SEG_A,0.45,-1,HistGradientBoosting
3,-1,0.780160,0.219840,0,SEG_A,0.45,-1,HistGradientBoosting
4,-1,0.932220,0.067780,0,SEG_A,0.45,-1,HistGradientBoosting


In [25]:
# %%
manifest_path = TENSOR_DIR / "tensor_manifest.csv"

manifest_df = pd.read_csv(manifest_path)

print("Manifest:", manifest_df.shape)
manifest_df.head()

Manifest: (20931, 3)


,NUEVO_ID,ATSEG_HCP,HCP_FOLD
0,1,NaN,-1
1,2,NaN,-1
2,3,SEG_A,1
3,4,NaN,-1
4,5,NaN,-1


In [26]:
# %%
manifest_unlabeled = manifest_df.loc[unlabeled_mask.numpy()].reset_index(drop=True)

print("manifest_unlabeled:", manifest_unlabeled.shape)
print("unlabeled_predictions_df:", unlabeled_predictions_df.shape)

assert len(manifest_unlabeled) == len(unlabeled_predictions_df)

manifest_unlabeled: (9032, 3)
unlabeled_predictions_df: (9032, 8)


In [27]:
# %%
cols_to_add = ["NUEVO_ID"]

# Si también quieres conservar columnas extra si existen:
for col in ["ATSEG_HCP", "HCP_FOLD"]:
    if col in manifest_df.columns and col not in cols_to_add:
        cols_to_add.append(col)

unlabeled_predictions_with_id_df = pd.concat(
    [
        manifest_unlabeled[cols_to_add].reset_index(drop=True),
        unlabeled_predictions_df.reset_index(drop=True)
    ],
    axis=1
)

unlabeled_predictions_with_id_df.head()

,NUEVO_ID,ATSEG_HCP,HCP_FOLD,true_label_encoded,prob_SEG_A,prob_SEG_BC,pred_binary_encoded,pred_binary_label,decision_threshold,hcp_fold,model_name
0,1,NaN,-1,-1,0.889189,0.110811,0,SEG_A,0.45,-1,HistGradientBoosting
1,2,NaN,-1,-1,0.909169,0.090831,0,SEG_A,0.45,-1,HistGradientBoosting
2,4,NaN,-1,-1,0.910450,0.089550,0,SEG_A,0.45,-1,HistGradientBoosting
3,5,NaN,-1,-1,0.780160,0.219840,0,SEG_A,0.45,-1,HistGradientBoosting
4,6,NaN,-1,-1,0.932220,0.067780,0,SEG_A,0.45,-1,HistGradientBoosting


In [28]:
# %%
unlabeled_predictions_with_id_df["pred_binary_label"].value_counts()

pred_binary_label
SEG_A     8399
SEG_BC     633
Name: count, dtype: int64

In [29]:
# %%
unlabeled_predictions_with_id_df["pred_binary_label"].value_counts(normalize=True) * 100

pred_binary_label
SEG_A     92.991585
SEG_BC     7.008415
Name: proportion, dtype: float64

In [30]:
# %%
unlabeled_predictions_with_id_df[["prob_SEG_A", "prob_SEG_BC"]].describe()

,prob_SEG_A,prob_SEG_BC
count,9032.000000,9032.000000
mean,0.840028,0.159972
std,0.177819,0.177819
min,0.001723,0.031565
25%,0.856335,0.076391
50%,0.904689,0.095311
75%,0.923609,0.143665
max,0.968435,0.998277


In [32]:
# %%
OUTPUT_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

unlabeled_output_path = OUTPUT_DIR / "unlabeled_predictions_binary_segA_vs_segBC_with_hcp_id.csv"

unlabeled_predictions_with_id_df.to_csv(
    unlabeled_output_path,
    index=False
)

print("Saved to:", unlabeled_output_path)

Saved to: C:\Users\omarl\Downloads\pfizer_outputs\unlabeled_predictions_binary_segA_vs_segBC_with_hcp_id.csv


In [7]:
# %%
# Original labels:
# 0 = SEG_A
# 1 = SEG_B
# 2 = SEG_C

y_test_original = y_test.numpy()

# Binary labels:
# 0 = SEG_A
# 1 = SEG_B/C
y_test_binary = (y_test_original != 0).astype(int)

pd.Series(y_test_binary).value_counts().sort_index()

0    1281
1    1099
Name: count, dtype: int64

In [8]:
# %%
X_test_flat = X_test.numpy().reshape(X_test.shape[0], -1)

print("X_test_flat:", X_test_flat.shape)

X_test_flat: (2380, 5590)


In [9]:
# %%
test_proba_bc = best_model.predict_proba(X_test_flat)[:, 1]

test_pred_binary = (test_proba_bc >= best_threshold).astype(int)

test_pred_label = np.where(
    test_pred_binary == 0,
    "SEG_A",
    "SEG_BC"
)

true_binary_label = np.where(
    y_test_binary == 0,
    "SEG_A",
    "SEG_BC"
)

true_original_label = np.where(
    y_test_original == 0,
    "SEG_A",
    np.where(y_test_original == 1, "SEG_B", "SEG_C")
)

In [10]:
# %%
test_predictions_df = pd.DataFrame({
    "true_original_label": true_original_label,
    "true_original_encoded": y_test_original,
    "true_binary_label": true_binary_label,
    "true_binary_encoded": y_test_binary,
    "prob_SEG_A": 1 - test_proba_bc,
    "prob_SEG_BC": test_proba_bc,
    "pred_binary_label": test_pred_label,
    "pred_binary_encoded": test_pred_binary,
    "decision_threshold": best_threshold,
    "hcp_fold": fold_test.numpy(),
    "model_name": best_model_name
})

test_predictions_df.head()

,true_original_label,true_original_encoded,true_binary_label,true_binary_encoded,prob_SEG_A,prob_SEG_BC,pred_binary_label,pred_binary_encoded,decision_threshold,hcp_fold,model_name
0,SEG_A,0,SEG_A,0,0.843814,0.156186,SEG_A,0,0.45,3,HistGradientBoosting
1,SEG_A,0,SEG_A,0,0.643654,0.356346,SEG_A,0,0.45,3,HistGradientBoosting
2,SEG_A,0,SEG_A,0,0.774528,0.225472,SEG_A,0,0.45,3,HistGradientBoosting
3,SEG_B,1,SEG_BC,1,0.062458,0.937542,SEG_BC,1,0.45,3,HistGradientBoosting
4,SEG_A,0,SEG_A,0,0.900509,0.099491,SEG_A,0,0.45,3,HistGradientBoosting


In [11]:
# %%
manifest_path = TENSOR_DIR / "tensor_manifest.csv"

if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    
    # Primero filtrar manifest a labeled y luego test fold,
    # para que quede alineado con X_test/y_test.
    manifest_labeled = manifest_df.loc[labeled_mask.numpy()].reset_index(drop=True)
    manifest_test = manifest_labeled.loc[test_mask.numpy()].reset_index(drop=True)
    
    test_predictions_df = pd.concat(
        [
            manifest_test.reset_index(drop=True),
            test_predictions_df.reset_index(drop=True)
        ],
        axis=1
    )

test_predictions_df.head()

,NUEVO_ID,ATSEG_HCP,HCP_FOLD,true_original_label,true_original_encoded,true_binary_label,true_binary_encoded,prob_SEG_A,prob_SEG_BC,pred_binary_label,pred_binary_encoded,decision_threshold,hcp_fold,model_name
0,20,SEG_A,3,SEG_A,0,SEG_A,0,0.843814,0.156186,SEG_A,0,0.45,3,HistGradientBoosting
1,47,SEG_A,3,SEG_A,0,SEG_A,0,0.643654,0.356346,SEG_A,0,0.45,3,HistGradientBoosting
2,63,SEG_A,3,SEG_A,0,SEG_A,0,0.774528,0.225472,SEG_A,0,0.45,3,HistGradientBoosting
3,69,SEG_B,3,SEG_B,1,SEG_BC,1,0.062458,0.937542,SEG_BC,1,0.45,3,HistGradientBoosting
4,76,SEG_A,3,SEG_A,0,SEG_A,0,0.900509,0.099491,SEG_A,0,0.45,3,HistGradientBoosting


In [12]:
# %%
OUTPUT_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_output_path = OUTPUT_DIR / "test_predictions_binary_segA_vs_segBC.csv"

test_predictions_df.to_csv(test_output_path, index=False)

print("Saved test predictions to:", test_output_path)

Saved test predictions to: C:\Users\omarl\Downloads\pfizer_outputs\test_predictions_binary_segA_vs_segBC.csv


In [13]:
OUTPUT_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_df = pd.read_csv(TENSOR_DIR / "tensor_manifest.csv")

manifest_labeled = manifest_df.loc[labeled_mask.numpy()].reset_index(drop=True)
manifest_test = manifest_labeled.loc[test_mask.numpy()].reset_index(drop=True)

assert len(manifest_test) == len(test_predictions_df)

test_predictions_with_id_df = pd.concat(
    [
        manifest_test[["NUEVO_ID"]].reset_index(drop=True),
        test_predictions_df.reset_index(drop=True)
    ],
    axis=1
)

test_predictions_with_id_df.to_csv(
    OUTPUT_DIR / "test_predictions_binary_segA_vs_segBC_with_hcp_id.csv",
    index=False
)

test_predictions_with_id_df.head()

,NUEVO_ID,NUEVO_ID,ATSEG_HCP,HCP_FOLD,true_original_label,true_original_encoded,true_binary_label,true_binary_encoded,prob_SEG_A,prob_SEG_BC,pred_binary_label,pred_binary_encoded,decision_threshold,hcp_fold,model_name
0,20,20,SEG_A,3,SEG_A,0,SEG_A,0,0.843814,0.156186,SEG_A,0,0.45,3,HistGradientBoosting
1,47,47,SEG_A,3,SEG_A,0,SEG_A,0,0.643654,0.356346,SEG_A,0,0.45,3,HistGradientBoosting
2,63,63,SEG_A,3,SEG_A,0,SEG_A,0,0.774528,0.225472,SEG_A,0,0.45,3,HistGradientBoosting
3,69,69,SEG_B,3,SEG_B,1,SEG_BC,1,0.062458,0.937542,SEG_BC,1,0.45,3,HistGradientBoosting
4,76,76,SEG_A,3,SEG_A,0,SEG_A,0,0.900509,0.099491,SEG_A,0,0.45,3,HistGradientBoosting


In [14]:
# %%
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test_binary, test_pred_binary))
print("Macro F1:", f1_score(y_test_binary, test_pred_binary, average="macro"))
print("Weighted F1:", f1_score(y_test_binary, test_pred_binary, average="weighted"))
print("ROC AUC:", roc_auc_score(y_test_binary, test_proba_bc))

print(classification_report(
    y_test_binary,
    test_pred_binary,
    target_names=["SEG_A", "SEG_BC"]
))

Accuracy: 0.7722689075630252
Macro F1: 0.7703775750550399
Weighted F1: 0.7719711978163979
ROC AUC: 0.8518300292864351
              precision    recall  f1-score   support

       SEG_A       0.78      0.80      0.79      1281
      SEG_BC       0.76      0.74      0.75      1099

    accuracy                           0.77      2380
   macro avg       0.77      0.77      0.77      2380
weighted avg       0.77      0.77      0.77      2380



In [15]:
from huggingface_hub import login

login()

In [16]:
import shutil
HF_EXPORT_DIR = MODEL_DIR / "hf_binary_segA_vs_segBC"
HF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# Model
shutil.copy(
    MODEL_DIR / "best_binary_segA_vs_segBC.joblib",
    HF_EXPORT_DIR / "best_binary_segA_vs_segBC.joblib"
)

# Metadata
shutil.copy(
    MODEL_DIR / "model_metadata.json",
    HF_EXPORT_DIR / "model_metadata.json"
)

# Optional: validation threshold comparison
threshold_file = MODEL_DIR / "binary_model_threshold_comparison_validation.csv"

if threshold_file.exists():
    shutil.copy(
        threshold_file,
        HF_EXPORT_DIR / "binary_model_threshold_comparison_validation.csv"
    )

# Optional: test predictions CSV
predictions_file = OUTPUT_DIR / "test_predictions_binary_segA_vs_segBC_with_hcp_id.csv"

if predictions_file.exists():
    shutil.copy(
        predictions_file,
        HF_EXPORT_DIR / "test_predictions_binary_segA_vs_segBC_with_hcp_id.csv"
    )

In [17]:
with open(MODEL_DIR / "model_metadata.json", "r") as f:
    metadata = json.load(f)

best_model_name = metadata.get("best_model_name", "Unknown")
best_threshold = metadata.get("best_threshold", "Unknown")
test_metrics = metadata.get("test_metrics", {})

In [18]:
readme_text = f"""
---
license: other
tags:
- binary-classification
- healthcare
- physician-segmentation
- scikit-learn
- xgboost
---

# Binary SEG_A vs SEG_B/C Classifier

This repository contains the selected binary classifier for the first stage of a hierarchical physician segmentation strategy.

## Task

Binary classification:

- `0`: SEG_A
- `1`: SEG_B/C

The model predicts whether a physician belongs to `SEG_A` or should be routed to the second-stage `SEG_B` vs `SEG_C` classifier.

## Selected Model

Best model: `{best_model_name}`  
Decision threshold for SEG_B/C: `{best_threshold}`

## Files

- `best_binary_segA_vs_segBC.joblib`: trained model
- `model_metadata.json`: model configuration and selected threshold
- `binary_model_threshold_comparison_validation.csv`: validation threshold comparison
- `test_predictions_binary_segA_vs_segBC_with_hcp_id.csv`: test-set predictions with HCP ID

## Notes

The model uses flattened temporal tensors as input. Each physician is represented by weekly behavior across multiple features.

The prediction probability `prob_SEG_BC` can be used to decide whether a physician should be classified as `SEG_A` or passed to the next B/C decision model.
"""

with open(HF_EXPORT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

In [20]:
from huggingface_hub import create_repo, upload_folder

HF_USERNAME = "omarleosamman"
REPO_NAME = "pfizer-binary-segA-vs-segBC"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=True,
    exist_ok=True
)

upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(HF_EXPORT_DIR),
    commit_message="Upload binary SEG_A vs SEG_BC classifier"
)

print(f"Uploaded to: https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to: https://huggingface.co/omarleosamman/pfizer-binary-segA-vs-segBC


In [21]:
from huggingface_hub import create_repo, upload_folder

ORG_NAME = "pfizer-project-team"   # cambia esto
REPO_NAME = "binary-segA-vs-segBC"
REPO_ID = f"{ORG_NAME}/{REPO_NAME}"

create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=True,
    exist_ok=True
)

upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=str(HF_EXPORT_DIR),
    commit_message="Upload binary SEG_A vs SEG_BC classifier"
)

print(f"https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

https://huggingface.co/pfizer-project-team/binary-segA-vs-segBC
